# 03 — Recommendation baselines

Examine temporal validation and the incremental, fixed-score strategy design.

**Executed artifact:** reusable transformations live in `src/` and `sql/`; this notebook reads compact, versioned evidence rather than reprocessing 230 million events interactively.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
def report(name): return json.loads((ROOT/'reports'/name).read_text())

In [2]:
p=json.loads((ROOT/'config/protocol.json').read_text())
p

{'seed': 20260831,
 'history_events': 5,
 'horizon_ms': 86400000,
 'recent_days': 7,
 'neighbor_count': 30,
 'minimum_pair_support': 2,
 'maximum_block_items': 10,
 'popular_candidate_count': 200,
 'recommendation_count': 20,
 'recency_half_life_hours': 6,
 'action_weights': [1, 3, 2],
 'popularity_weight': 0.05,
 'bootstrap_replicates': 1000,
 'evaluation_batch_sessions': 2000,
 'duckdb_memory_limit': '512MB',
 'duckdb_threads': 1}

In [3]:
v=report('results_validation.json')
pd.DataFrame([{'strategy':k.replace('_recall',''),'recall@20':x['estimate'],'lower':x['ci95'][0],'upper':x['ci95'][1]} for k,x in v['metrics'].items() if k.endswith('_recall')])

,strategy,recall@20,lower,upper
0,global,0.006190,0.005903,0.006466
1,recent,0.006213,0.005929,0.006500
2,repeat,0.464663,0.462950,0.466494
3,r,0.464663,0.462950,0.466494
4,ra,0.527162,0.525470,0.528978
5,c,0.524310,0.522632,0.526035


The validation week checks temporal replication and pipeline behavior. It does not select a favorable target, candidate rule or metric.

## KEY FINDINGS

Popularity and recent repeat provide nontrivial benchmarks. Incremental strategies share the same candidate pool after repetition so the ablation isolates information used in ranking.

## LIMITATIONS

Hand-set scores are transparent heuristics, not calibrated probabilities; validation is one week.

## NEXT STEP

Evaluate the untouched test week with paired uncertainty and diagnose retrieval versus ranking.